In [ ]:
# Connect to Google Drive
from google.colab import drive
drive.mount('/gdrive')
%cd /gdrive/MyDrive/ai4culture

Mounted at /gdrive
/gdrive/MyDrive/ai4culture


In [ ]:
# Step 2: Import necessary libraries
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense
import os

# Step 3: Set the path to your dataset folder
# Replace 'your_dataset_folder_path' with the actual path to your dataset folder in Google Drive.
# For example: '/content/drive/MyDrive/your_dataset_folder'
dataset_dir = 'small'  # <-- Insert your dataset path here

# Step 4: Define image parameters and batch size
img_height, img_width = 150, 150  # You can adjust the target size if needed
batch_size = 32

# Step 5: Create data generators with a validation split
datagen = ImageDataGenerator(
    rescale=1./255,         # Normalize pixel values to [0,1]
    validation_split=0.2    # 20% of the data will be used for validation
)

# Training data generator (80% of the data)
train_generator = datagen.flow_from_directory(
    directory=dataset_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary',    # Binary classification
    subset='training',
    shuffle=True
)

# Validation data generator (20% of the data)
validation_generator = datagen.flow_from_directory(
    directory=dataset_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary',
    subset='validation',
    shuffle=True
)

# Step 6: Build a simple Convolutional Neural Network (CNN) model
model = Sequential([
    # First convolution block
    Conv2D(32, (3, 3), activation='relu', input_shape=(img_height, img_width, 3)),
    MaxPooling2D(pool_size=(2, 2)),

    # Second convolution block
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    # Third convolution block
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    # Flatten and Dense layers for classification
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # Sigmoid activation for binary classification
])

# Step 7: Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Display the model architecture
model.summary()

# Step 8: Train the model
epochs = 10  # You can adjust the number of epochs
history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=validation_generator
)

# Step 9: (Optional) Save the trained model
model.save('old_painting_age_classifier.h5')
print("Model saved as old_painting_age_classifier.h5")

# Step 10: Define a function to predict the class of a new painting image
import numpy as np
from tensorflow.keras.preprocessing import image

def predict_painting(img_path):
    """
    Predicts whether a painting is aged or restored/original.

    Args:
        img_path (str): Path to the image file.

    Returns:
        None. Prints the prediction result.
    """
    # Load and preprocess the image
    img = image.load_img(img_path, target_size=(img_height, img_width))
    img_array = image.img_to_array(img) / 255.0  # Normalize pixel values
    img_array = np.expand_dims(img_array, axis=0)  # Create batch dimension

    # Make prediction
    prediction = model.predict(img_array)

    # Interpret prediction (threshold is 0.5; adjust if necessary)
    if prediction[0] < 0.5:
        print("The painting is likely restored/original.")
    else:
        print("The painting is likely aged.")

# Example usage:
# After training, you can test the function by uploading a test image.
# Uncomment the line below and replace '/content/your_test_image.jpg' with your image path.
# predict_painting('/content/your_test_image.jpg')

Found 106 images belonging to 2 classes.
Found 26 images belonging to 2 classes.


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 148, 148, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 74, 74, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 72, 72, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 36, 36, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 34, 34, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 17, 17, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 36992)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 512)                 │      18,940,416 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │             513 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 19,034,177 (72.61 MB)

 Trainable params: 19,034,177 (72.61 MB)

 Non-trainable params: 0 (0.00 B)

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 30s 7s/step - accuracy: 0.4603 - loss: 0.9344 - val_accuracy: 0.5000 - val_loss: 0.6922
Epoch 2/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 21s 2s/step - accuracy: 0.6355 - loss: 0.6506 - val_accuracy: 0.5000 - val_loss: 0.6836
Epoch 3/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 10s 3s/step - accuracy: 0.4866 - loss: 0.6885 - val_accuracy: 0.6923 - val_loss: 0.6614
Epoch 4/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 10s 2s/step - accuracy: 0.5982 - loss: 0.6471 - val_accuracy: 0.6538 - val_loss: 0.6416
Epoch 5/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 9s 2s/step - accuracy: 0.6073 - loss: 0.6239 - val_accuracy: 0.6538 - val_loss: 0.6354
Epoch 6/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 10s 2s/step - accuracy: 0.6640 - loss: 0.6159 - val_accuracy: 0.6923 - val_loss: 0.6964
Epoch 7/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 10s 2s/step - accuracy: 0.7006 - loss: 0.5799 - val_accuracy: 0.6538 - val_loss: 0.6491
Epoch 8/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 9s 2s/step - accuracy: 0.7452 - loss: 0.5370 - val_accuracy: 0.7308 - val_loss: 0.6623
Epoch 9/10

Model saved as old_painting_age_classifier.h5


In [ ]:
import numpy as np

avg_train_acc = np.mean(history.history['accuracy'])
avg_train_loss = np.mean(history.history['loss'])
avg_val_acc = np.mean(history.history['val_accuracy'])
avg_val_loss = np.mean(history.history['val_loss'])

print("Average Training Accuracy:", avg_train_acc)
print("Average Training Loss:", avg_train_loss)
print("Average Validation Accuracy:", avg_val_acc)
print("Average Validation Loss:", avg_val_loss)


Average Training Accuracy: 0.6452830135822296
Average Training Loss: 0.6364680528640747
Average Validation Accuracy: 0.6576923072338104
Average Validation Loss: 0.667972183227539


In [ ]:
# predict_painting('fernando-botero_luncheon-on-the-grass.jpg')

In [ ]:
# Step 2: Import necessary libraries
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense
import os

# Step 3: Set the path to your dataset folder
# Replace 'your_dataset_folder_path' with the actual path to your dataset folder in Google Drive.
# For example: '/content/drive/MyDrive/your_dataset_folder'
dataset_dir = 'PaintingDataset'  # <-- Insert your dataset path here

# Step 4: Define image parameters and batch size
img_height, img_width = 150, 150  # You can adjust the target size if needed
batch_size = 32

# Step 5: Create data generators with a validation split
datagen = ImageDataGenerator(
    rescale=1./255,         # Normalize pixel values to [0,1]
    validation_split=0.2    # 20% of the data will be used for validation
)

# Training data generator (80% of the data)
train_generator = datagen.flow_from_directory(
    directory=dataset_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary',    # Binary classification
    subset='training',
    shuffle=True
)

# Validation data generator (20% of the data)
validation_generator = datagen.flow_from_directory(
    directory=dataset_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary',
    subset='validation',
    shuffle=True
)

# Step 6: Build a simple Convolutional Neural Network (CNN) model
model = Sequential([
    # First convolution block
    Conv2D(32, (3, 3), activation='relu', input_shape=(img_height, img_width, 3)),
    MaxPooling2D(pool_size=(2, 2)),

    # Second convolution block
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    # Third convolution block
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    # Flatten and Dense layers for classification
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # Sigmoid activation for binary classification
])

# Step 7: Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Display the model architecture
model.summary()

# Step 8: Train the model
epochs = 10  # You can adjust the number of epochs
history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=validation_generator
)

# Step 9: (Optional) Save the trained model
model.save('painting_age_classifier.h5')
print("Model saved as painting_age_classifier.h5")

# Step 10: Define a function to predict the class of a new painting image
import numpy as np
from tensorflow.keras.preprocessing import image

def predict_painting(img_path):
    """
    Predicts whether a painting is aged or restored/original.

    Args:
        img_path (str): Path to the image file.

    Returns:
        None. Prints the prediction result.
    """
    # Load and preprocess the image
    img = image.load_img(img_path, target_size=(img_height, img_width))
    img_array = image.img_to_array(img) / 255.0  # Normalize pixel values
    img_array = np.expand_dims(img_array, axis=0)  # Create batch dimension

    # Make prediction
    prediction = model.predict(img_array)

    # Interpret prediction (threshold is 0.5; adjust if necessary)
    if prediction[0] < 0.5:
        print("The painting is likely restored/original.")
    else:
        # print("The painting is likely aged.")
        # print("The painting is likely to have a Patina due to aging.")
        print("The artwork exhibits signs of aging, resulting in a modified appearance.")

# Example usage:
# After training, you can test the function by uploading a test image.
# Uncomment the line below and replace '/content/your_test_image.jpg' with your image path.
# predict_painting('/content/your_test_image.jpg')

Found 480 images belonging to 2 classes.
Found 120 images belonging to 2 classes.


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)                    │ (None, 148, 148, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 74, 74, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_4 (Conv2D)                    │ (None, 72, 72, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_4 (MaxPooling2D)       │ (None, 36, 36, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_5 (Conv2D)                    │ (None, 34, 34, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_5 (MaxPooling2D)       │ (None, 17, 17, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_1 (Flatten)                  │ (None, 36992)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 512)                 │      18,940,416 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 1)                   │             513 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 19,034,177 (72.61 MB)

 Trainable params: 19,034,177 (72.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 88s 6s/step - accuracy: 0.4848 - loss: 1.1897 - val_accuracy: 0.6500 - val_loss: 0.6894
Epoch 2/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 34s 2s/step - accuracy: 0.5616 - loss: 0.6907 - val_accuracy: 0.6250 - val_loss: 0.6782
Epoch 3/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 37s 2s/step - accuracy: 0.5826 - loss: 0.6690 - val_accuracy: 0.5917 - val_loss: 0.6453
Epoch 4/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 33s 2s/step - accuracy: 0.6184 - loss: 0.6476 - val_accuracy: 0.6750 - val_loss: 0.5999
Epoch 5/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 34s 2s/step - accuracy: 0.6454 - loss: 0.6270 - val_accuracy: 0.6500 - val_loss: 0.5680
Epoch 6/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 41s 3s/step - accuracy: 0.6347 - loss: 0.5802 - val_accuracy: 0.7417 - val_loss: 0.4699
Epoch 7/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 34s 2s/step - accuracy: 0.6942 - loss: 0.5220 - val_accuracy: 0.7250 - val_loss: 0.4639
Epoch 8/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 37s 2s/step - accuracy: 0.7186 - loss: 0.5089 - val_accuracy: 0.7833 - val_loss:

Model saved as painting_age_classifier.h5


In [ ]:
import numpy as np

avg_train_acc = np.mean(history.history['accuracy'])
avg_train_loss = np.mean(history.history['loss'])
avg_val_acc = np.mean(history.history['val_accuracy'])
avg_val_loss = np.mean(history.history['val_loss'])

print("Average Training Accuracy:", avg_train_acc)
print("Average Training Loss:", avg_train_loss)
print("Average Validation Accuracy:", avg_val_acc)
print("Average Validation Loss:", avg_val_loss)

Average Training Accuracy: 0.6558333337306976
Average Training Loss: 0.5993924379348755
Average Validation Accuracy: 0.7008333325386047
Average Validation Loss: 0.5353918135166168


In [ ]:
# predict_painting('katsushika-hokusai_fukagawa-mannen-bashi-shita high.jpg')

In [ ]:
# predict_painting('test.jpg')

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

def predict_probability(img_path):
    # Load and preprocess the image
    img = image.load_img(img_path, target_size=(img_height, img_width))
    img_array = image.img_to_array(img) / 255.0  # normalize the image
    img_array = np.expand_dims(img_array, axis=0)  # add batch dimension

    # Predict and return the numerical value (probability)
    prob = model.predict(img_array)
    return prob[0][0]

# Example usage:
img_path = 'pierre-puvis-de-chavannes_the-poor-fisherman-1.jpg'
probability = predict_probability(img_path)
print(f"Predicted probability: {probability:.2f}")
predicted_class = 'The artwork exhibits signs of aging, resulting in a modified appearance.' if probability >= 0.38 else 'The painting is likely restored/original.'
print(f"Painting status: {predicted_class}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
Predicted probability: 0.45
Painting status: The artwork exhibits signs of aging, resulting in a modified appearance.


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

def predict_probability(img_path):
    # Load and preprocess the image
    img = image.load_img(img_path, target_size=(img_height, img_width))
    img_array = image.img_to_array(img) / 255.0  # normalize the image
    img_array = np.expand_dims(img_array, axis=0)  # add batch dimension

    # Predict and return the numerical value (probability)
    prob = model.predict(img_array)
    return prob[0][0]

# Example usage:
img_path = 'tsukioka-restored.jpg'
probability = predict_probability(img_path)
print(f"Predicted probability: {probability:.2f}")
predicted_class = 'The artwork exhibits signs of aging, resulting in a modified appearance.' if probability >= 0.20 else 'The painting is likely restored/original.'
print(f"Painting status: {predicted_class}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
Predicted probability: 0.72
Painting status: The artwork exhibits signs of aging, resulting in a modified appearance.


#               PATINA - DE/Color of Time

In [ ]:
predict_painting('test.jpg')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
The artwork exhibits signs of aging, resulting in a modified appearance.


In [ ]:
predict_painting('')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
The artwork exhibits signs of aging, resulting in a modified appearance.
